# daftar in a notebook

Notebooks are where provenance dies. The git commit is close to meaningless —
a notebook is one file whose cells ran in an order nobody recorded. Cells get
edited and re-run, so the code that made a figure may no longer exist anywhere.
And on Colab there is no repository at all.

daftar records the two things that actually identify a notebook result:

- **`code.cell_sha256`** — the source of the cell that ran, captured *before*
  execution, so it survives you editing the cell afterwards.
- **`code.session_history_sha256`** — every cell executed before it. A result
  depends on the whole session, and nothing else records that.

Run the cells in order, then run the **out-of-order** section at the end.

In [ ]:
!pip install -q daftar

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 4.9 MB/s eta 0:00:00


In [ ]:
%load_ext daftar
import daftar
print("daftar", daftar.__version__)

daftar: %%daftar cell magic registered
daftar 1.0.0


## 1. A tracked cell

`%%daftar <label> seed=<n>` wraps the cell. A `run` object is injected — no import needed.

In [ ]:
import numpy as np
scale = 1.0

In [ ]:
%%daftar montecarlo seed=42
rng = np.random.default_rng(0)
vals = rng.normal(size=100_000) * scale
run.log_result("mean", float(vals.mean()))
run.log_result("std", float(vals.std()))
print("mean", vals.mean())

mean -0.0009082507731206114
daftar: recorded r-f1d64487


## 2. What was recorded

In [ ]:
store = daftar.RunStore()
m = store.load(store.list()[0].run_id)
for k in m.ordered_keys:
    if k.split(".")[0] in ("code", "seed", "result"):
        print(f"{k:<34} {m.fields[k][:60]}")

code.argv                          /usr/local/lib/python3.13/dist-packages/colab_kernel_launche
code.cell_lines                    5
code.cell_sha256                   1c170d388dc2d5a7
code.cell_source                   rng = np.random.default_rng(0)
vals = rng.normal(size=100_00
code.entrypoint                    notebook::cell[1c170d388dc2d5a7]
code.kernel                        colab
code.notebook                      true
code.notebook_path                 1sGIB4YXnho0WXiX5rCUwRCIzgT-P60PN
code.session_history_sha256        b09f4653fa1d7d27
code.session_n_cells               4
code.session_n_redefined           0
code.vcs                           none
seed.numpy_legacy                  true
seed.python_random                 true
seed.pythonhashseed_env            unset
seed.value                         42
seed.was_explicit                  true
result.mean                        -0.0009082507731206114
result.std                         1.0001285040531447


## 3. The thing no other tool catches

Change a variable an earlier cell defined, then re-run **the identical cell**.
Same code, same seed, same everything on disk — different answer.

In [ ]:
scale = 3.0   # an upstream cell changes

In [ ]:
%%daftar montecarlo seed=42
rng = np.random.default_rng(0)
vals = rng.normal(size=100_000) * scale
run.log_result("mean", float(vals.mean()))
run.log_result("std", float(vals.std()))
print("mean", vals.mean())

mean -0.002724752319361834
daftar: recorded r-4ba64c6a


In [ ]:
runs = daftar.RunStore().list(limit=2)
a, b = store.load(runs[1].run_id), store.load(runs[0].run_id)
print(daftar.render_diff(daftar.diff_manifests(a, b)))

--- r-f1d64487  montecarlo  2026-09-10T12:22:08.804991+00:00
+++ r-4ba64c6a  montecarlo  2026-09-10T12:22:18.836825+00:00

candidate causes (5)
  code.session_history_sha256  b09f4653fa1d7d27  ->  27c179a5f8fd334a
  code.session_n_cells         4  ->  7
  code.session_n_redefined     0  ->  1
  code.session_redefined       (absent)  ->  [scale]
  code.session_stale_risk      (absent)  ->  true

observed effects (2)
  result.mean                  -0.0009082507731206114  ->  -0.002724752319361834
  result.std                   1.0001285040531447  ->  3.000385512159434

7 meaningful field(s) differ, 70 identical  (+2 cost/metadata, hidden -- use --all)

Results differ, and so do things that could explain it. The candidate causes below are where to look.


The cell hash is **identical** — it is the same code. What differs is
`code.session_history_sha256`: the session state that produced the number.

Without that field this would have been reported as *nondeterministic*, which
would have been wrong and would have sent you looking for a seeding bug that
does not exist.

## 4. Without the magic

`daftar.track()` detects the notebook by itself. Existing code needs no changes
— it simply gains the notebook fields.

In [ ]:
with daftar.track("plain-track", params={"n": 1000}, seed=7) as run:
    x = np.random.default_rng(run.seed).normal(size=1000)
    run.log_result("mean", float(x.mean()))

m = store.load(run.run_id)
print("entrypoint       ", m.get("code.entrypoint"))
print("cell hash        ", m.get("code.cell_sha256"))
print("session history  ", m.get("code.session_history_sha256"))
print("cells before this", m.get("code.session_n_cells"))

entrypoint        notebook::cell[f629a4b81ea94125]
cell hash         f629a4b81ea94125
session history   3582d016fd6639cd
cells before this 8


## 5. Hand it to someone else

The bundle contains `cell.py` — the source that actually ran. On Colab, where
the VM is ephemeral and nothing is committed, this is often the only surviving
copy of the code that produced the result.

In [ ]:
bundle = daftar.export_bundle(m, "run.zip", store=store)
import zipfile
print(zipfile.ZipFile(bundle).namelist())
print(daftar.plan_replay(m, check_current=False).render())

['README.md', 'manifest.json', 'fields.tsv']
Replay plan for r-15d4f8de

  entrypoint   notebook::cell[f629a4b81ea94125]
  commit       
  seed         7

  parameters
    n  1000

  No blockers. The recorded state is recoverable.

  warnings
    - run was not made inside a git repository
    - run happened in a notebook cell (sha256 f629a4b81ea94125) after 8 other cells had executed in that session; the result depends on all of them and the .ipynb on disk records file order, not execution order
    - no repository, so the notebook itself is not version-pinned; the exported bundle contains the cell source that actually ran
    - torch was installed from a local path (local:https://download.pytorch.org/whl/cpu/torch-2.11.0%2Bcpu-cp313-cp313-manylinux_2_28_x86_64.whl); its version string does not identify the code and no one else can fetch it

  To reproduce:
    # 46 packages recorded; see env.* in the manifest
    pip install IPython==7.34.0 anywidget==0.9.21 backcall==0.2.0 certifi==2

## 6. Try it on your own work

Add one line to a notebook you already have:

```python
with daftar.track("my-experiment", seed=42) as run:
    ...            # your existing code, unchanged
    run.log_result("accuracy", acc)
```

Then run it twice on different days and `daftar diff` the two.